# Ejercicio de Hold Out
![logo Itq](img/ITQ.png)

**Nombre:** Freddy Guevara

**Fecha** *12-11-2026*

**Hold Out:** técnica de validación que divide el conjunto de datos en dos partes: una para entrenar el modelo y otra, que el modelo no ha visto, para evaluarlo. Así podemos medir qué tan bien generaliza el modelo con datos nuevos.

**Link Repositorio** https://github.com/Fguevara92/Fund_Machin_Learn_IA_Noct_26_27/blob/main/Programa4.Ausentes.ipynb

---
### Importación de librerías
* **numpy**: nos permite trabajar con arreglos y contar las ocurrencias de cada clase.
* **datasets**: módulo de scikit-learn que trae conjuntos de datos de ejemplo, como Iris.
* **DummyClassifier**: clasificador muy simple que predice sin aprender de los atributos. Sirve como modelo base para comparar.
* **train_test_split**: función que divide los datos en entrenamiento y prueba.
* **StandardScaler**: estandariza los atributos para que tengan media 0 y desviación estándar 1.

In [1]:
import numpy as np
from sklearn import datasets
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

### Carga de datos
Cargamos el dataset **Iris**, que contiene medidas de flores (largo y ancho del sépalo y del pétalo). Mostramos los nombres de las tres especies que el modelo debe clasificar: *setosa*, *versicolor* y *virginica*.

In [2]:
# Carga de datos
iris = datasets.load_iris()
print(iris.target_names)

['setosa' 'versicolor' 'virginica']


### Exploración del conjunto de datos
Revisamos cuántas instancias y atributos tiene la tabla, qué valores toma la clase y cuántos registros hay de cada una.

In [3]:
# Mostrar características de la tabla de datos.
print("Tabla de datos: %d instancias y %d atributos" % (iris.data.shape[0], iris.data.shape[1]))
print("Valores de la clase:", set(iris.target))

# Cuantificamos el número de instancias que contiene el dataset por clase
valores, ocurrencias = np.unique(iris.target, return_counts=True)
print(valores, ocurrencias)

Tabla de datos: 150 instancias y 4 atributos
Valores de la clase: {np.int64(0), np.int64(1), np.int64(2)}
[0 1 2] [50 50 50]


**Interpretación:** el dataset tiene 150 instancias y 4 atributos. Las clases son 0, 1 y 2, con 50 registros cada una, así que el conjunto está **balanceado**.

### Partición externa: entrenamiento y test
* Dividimos los datos en **80% para entrenamiento** y **20% para test** con `test_size=0.2`.
* `random_state=42` fija la semilla para que la división sea siempre la misma y podamos repetir el resultado.
* El conjunto de test se guarda aparte y solo se usa al final para la evaluación definitiva.

In [4]:
# Test: hold-out split 80-20%. # Partición externa
X_training, X_test, y_training, y_test = train_test_split(iris.data, iris.target, test_size=0.2, random_state=42)
valores_test, ocur_test = np.unique(y_test, return_counts=True)
print('Test: ', 'clases:', valores_test, ' ocurrencias: ', ocur_test)

Test:  clases: [0 1 2]  ocurrencias:  [10  9 11]


### Estandarización
* `fit_transform()` **aprende** la media y la desviación del entrenamiento y lo transforma.
* En el test usamos solo `transform()`, con los valores aprendidos del entrenamiento.

Esto evita la **fuga de datos** (*data leakage*): el modelo no debe conocer nada del test antes de evaluarse.

In [5]:
# Estandarizar las características de entrenamiento y de test
standardizer = StandardScaler()
X_training = standardizer.fit_transform(X_training)
X_test = standardizer.transform(X_test)

### Partición interna: entrenamiento y validación
Del conjunto de entrenamiento separamos otro 20% para **validación**. Este conjunto nos permite probar y ajustar el modelo sin tocar el test.

In [6]:
# Validación: hold-out split 80-20%. # Partición interna
X_train, X_val, y_train, y_val = train_test_split(X_training, y_training, test_size=0.2, random_state=42)
valores_train, ocur_train = np.unique(y_train, return_counts=True)
print('Entrenamiento: ', ' clases:', valores_train, '  ocurrencias:', ocur_train)

valores_val, ocur_val = np.unique(y_val, return_counts=True)
print('Validation:    ', ' clases:', valores_val, '  ocurrencias:', ocur_val)

Entrenamiento:   clases: [0 1 2]   ocurrencias: [32 30 34]
Validation:      clases: [0 1 2]   ocurrencias: [ 8 11  5]


**Interpretación:** quedan 96 instancias para entrenar, 24 para validar y 30 para test. Las clases no quedan exactamente iguales en cada parte porque la división es aleatoria.

### Creación del modelo base
Creamos un `DummyClassifier` con `strategy='prior'`. Este modelo siempre predice la clase más frecuente del entrenamiento y no aprende de los atributos. Lo usamos como **punto de referencia**: cualquier modelo real debería superarlo.

In [7]:
# Construcción del objeto que contiene el algoritmo de aprendizaje.
clf = DummyClassifier(strategy='prior', random_state=42)

### Entrenamiento del modelo base
Con `fit()` entrenamos el clasificador usando los datos de entrenamiento.

In [8]:
# Entrenamiento del algoritmo de aprendizaje.
clf = clf.fit(X_train, y_train)

### Evaluación del modelo base
El método `score()` devuelve la **exactitud** (*accuracy*), es decir, el porcentaje de predicciones correctas. La calculamos en validación y en test.

In [9]:
# Evaluación del algoritmo de aprendizaje con el método "score" que devuelve directamente la métrica de 'accuracy'
val_accuracy = clf.score(X_val, y_val)
print("Exactitud en validación: ", val_accuracy)

test_accuracy = clf.score(X_test, y_test)
print("Exactitud en test: ", test_accuracy)

Exactitud en validación:  0.20833333333333334
Exactitud en test:  0.36666666666666664


**Interpretación:** la exactitud es de **20,8%** en validación y **36,7%** en test. Son valores bajos porque el modelo no aprende de los datos y solo acierta cuando la clase real coincide con la que siempre predice.

### Predicciones del modelo base
Con `predict()` obtenemos las clases que predice el modelo y las comparamos con las etiquetas reales de validación y de test.

In [10]:
# Obtenemos las predicciones sobre conjunto de validación y de test
y_pred_val = clf.predict(X_val)
print('Predicciones de validación ', y_pred_val)
print('Etiquetas reales validación', y_val)

y_pred_test = clf.predict(X_test)
print('\nPredicciones de test ', y_pred_val)
print('Etiquetas reales test', y_val)

Predicciones de validación  [2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]
Etiquetas reales validación [1 1 0 0 0 2 1 2 2 2 1 1 1 1 1 0 2 0 1 0 1 1 0 0]

Predicciones de test  [2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]
Etiquetas reales test [1 1 0 0 0 2 1 2 2 2 1 1 1 1 1 0 2 0 1 0 1 1 0 0]


### Modelo SVC
* **SVC** (*Support Vector Classifier*) es una máquina de vectores de soporte que busca la frontera que mejor separa las clases.
* El parámetro `C=0.5` controla la regularización: un valor más bajo hace el modelo más flexible frente a errores y ayuda a evitar el sobreajuste.
* Lo entrenamos y lo evaluamos en validación y en test.

In [11]:
# Aplicamos un ejemplo con un clasificador más complejo que el "dummyclassifier"
from sklearn.svm import SVC
svc = SVC(C=0.5) # Definimos algoritmo
svc.fit(X_train, y_train) # Entrenamos modelo

val_accuracy = svc.score(X_val, y_val) # Evaluamos modelo en validación
print('Exactitud en validación ', np.round(val_accuracy*100, 4), '%')

test_accuracy = svc.score(X_test, y_test) # Evaluamos modelo en test
print('Exactitud en test ', np.round(test_accuracy*100, 4), '%')

Exactitud en validación  91.6667 %
Exactitud en test  96.6667 %


**Interpretación:** el SVC obtiene **91,67%** en validación y **96,67%** en test, muy por encima del modelo base. Esto demuestra que sí aprende de los atributos de las flores.

### Guardar el modelo
* **pickle** permite guardar objetos de Python en un archivo para reutilizarlos sin volver a entrenar.
* **Path** (de `pathlib`) crea la carpeta `models` con `mkdir(exist_ok=True)`. Si la carpeta ya existe, no da error.
* Guardamos un **diccionario** con tres elementos:
  * `modelo`: el SVC entrenado.
  * `estandarizacion`: el `StandardScaler` ajustado con el entrenamiento. Es necesario porque los datos nuevos deben escalarse igual que los de entrenamiento antes de predecir.
  * `clases`: los nombres de las especies, para mostrar la predicción como texto y no como número.
* `'wb'` abre el archivo en modo escritura binaria y `pickle.dump()` guarda el diccionario.

In [12]:
# Guardar modelo
import pickle
from pathlib import Path

Path("models").mkdir(exist_ok=True)
with open('models/svc.pickle', 'wb') as fw:
    pickle.dump({
        "modelo":svc,
        "estandarizacion": standardizer,
        "clases": iris.target_names
    }, fw)

### Cargar el modelo guardado
* `'rb'` abre el archivo en modo lectura binaria y `pickle.load()` recupera el diccionario que guardamos antes.
* Del diccionario sacamos cada elemento por su clave:
  * `modelo`: el SVC ya entrenado, listo para hacer predicciones.
  * `estandarizador`: el `StandardScaler` ajustado con el entrenamiento, que usaremos para escalar los datos nuevos de la misma forma.
  * `clases`: los nombres de las especies, para mostrar la predicción como texto.

Así podemos usar el modelo en cualquier momento sin volver a entrenarlo.

In [13]:
#Cargamos el modelo guardado
with open("models/svc.pickle", "rb") as archivo:
    datos = pickle.load(archivo)

#Cargamos cada uno de los atributos
modelo = datos["modelo"]
estandarizador = datos["estandarizacion"]
clases = datos["clases"]

### Ingreso de datos por teclado
* Con `input()` pedimos al usuario las cuatro medidas de la flor en centímetros, y con `float()` las convertimos a número decimal.
* Con `np.array([[...]])` armamos el arreglo de entrada. Usamos **doble corchete** porque el modelo espera una tabla (filas y columnas), aunque solo sea una flor.
* Las medidas deben ir en el **mismo orden** que el dataset Iris: largo del sépalo, ancho del sépalo, largo del pétalo y ancho del pétalo. Si cambiamos el orden, el modelo interpreta mal los datos.
* Por último, estandarizamos la entrada con `transform()`, usando la media y la desviación aprendidas en el entrenamiento. Así los datos nuevos quedan en la misma escala con la que se entrenó el modelo.

In [14]:
#Ingresamos los datos por teclado
largo_sepalo = float(input("Ingrese el largo del sepalo (cm): "))
ancho_sepalo = float(input("Ingrese el ancho del sepalo (cm): "))
largo_petalo = float(input("Ingrese el largo del petalo (cm): "))
ancho_petalo = float(input("Ingrese el ancho del petalo (cm): "))

#Mantener el mismo orden de caracteristicas de entrada del data set Iris
entrada = np.array([[
    largo_sepalo,
    ancho_sepalo,
    largo_petalo,
    ancho_petalo
]])

#Imprimimos los datos de entrada
print("Los datos de entrada son: " , entrada)
#Transformamos los datos de entrada con la operación de estandarización
entrada_estandarizada = estandarizador.transform(entrada)

Ingrese el largo del sepalo (cm):  5.01
Ingrese el ancho del sepalo (cm):  3.42
Ingrese el largo del petalo (cm):  1.46
Ingrese el ancho del petalo (cm):  0.24


Los datos de entrada son:  [[5.01 3.42 1.46 0.24]]


### Visualizar la entrada estandarizada
Mostramos los datos de la flor después de estandarizarlos, para ver cómo quedaron antes de pasarlos al modelo.

In [15]:
print(entrada_estandarizada)

[[-0.97415946  0.80119649 -1.29892864 -1.2592088 ]]


**Interpretación:** las medidas ya no están en centímetros. Ahora indican cuántas desviaciones estándar se alejan de la media del entrenamiento:
* Un valor **cercano a 0** significa que la medida está cerca del promedio.
* Un valor **negativo** significa que está por debajo del promedio.
* Un valor **positivo** significa que está por encima del promedio.

Esta es la escala con la que se entrenó el modelo, por eso es necesario transformar los datos antes de predecir.

### Predicción con datos nuevos
* Con `predict()` el modelo clasifica la flor a partir de la entrada estandarizada. El resultado es un arreglo con un solo valor.
* Con `prediccion[0]` tomamos ese valor, y con `int()` lo convertimos a número entero para usarlo como índice.
* Ese número indica la clase: **0 = setosa**, **1 = versicolor** y **2 = virginica**.
* Con `clases[clase]` obtenemos el nombre de la especie, así el resultado es más fácil de entender.

In [16]:
#Generamos la nueva predicción
prediccion = modelo.predict(entrada_estandarizada)
clase = int(prediccion[0])
print("Clase predicha:", clase)
print("Especie predicha", clases[clase])

Clase predicha: 0
Especie predicha setosa


**Interpretación:** el modelo cargado clasificó la flor ingresada por teclado usando lo que aprendió en el entrenamiento, sin necesidad de volver a entrenarlo. El resultado se muestra como número de clase y como nombre de la especie.